# Phase 1 — Exploratory Data Analysis
## Online Retail II Dataset

**Sections**
1. Data loading & shape
2. Data quality & missing values
3. Revenue & quantity distributions
4. RFM analysis
5. Cohort retention
6. Category & geographic breakdown
7. Engineered feature distributions

## 1. Data Loading & Shape

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, calculate_rfm, engineer_features,
    build_cohort_matrix, build_master_customer_table
)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
print(f'shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df_raw.describe(include='all').T

## 2. Data Quality & Missing Values

In [ ]:
missing = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
missing_df[missing_df['missing_count'] > 0]['missing_pct'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Missing Value % by Column')
ax.set_xlabel('% missing')
plt.tight_layout()
plt.show()

In [ ]:
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, df_all = clean_data(df_raw)

## 3. Revenue & Quantity Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
revenue_clip = df_customers['revenue'].clip(upper=df_customers['revenue'].quantile(0.99))
axes[0].hist(revenue_clip, bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Revenue per Line (clipped at 99th pct)')
axes[0].set_xlabel('Revenue (£)')
qty_clip = df_customers['quantity'].clip(upper=df_customers['quantity'].quantile(0.99))
axes[1].hist(qty_clip, bins=60, color='coral', edgecolor='white')
axes[1].set_title('Quantity per Line (clipped at 99th pct)')
plt.tight_layout()
plt.show()

In [ ]:
monthly = (
    df_customers.assign(month=df_customers['invoice_date'].dt.to_period('M'))
    .groupby('month')['revenue'].sum().reset_index()
)
monthly['month'] = monthly['month'].astype(str)
fig = px.bar(monthly, x='month', y='revenue', title='Monthly Revenue (£)')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 4. RFM Analysis

In [ ]:
rfm = calculate_rfm(df_customers)
rfm.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].hist(rfm['recency'].clip(upper=rfm['recency'].quantile(0.99)), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Recency (days)')
axes[1].hist(rfm['frequency'].clip(upper=rfm['frequency'].quantile(0.99)), bins=60, color='seagreen', edgecolor='white')
axes[1].set_title('Frequency (orders)')
axes[2].hist(rfm['monetary'].clip(upper=rfm['monetary'].quantile(0.99)), bins=60, color='coral', edgecolor='white')
axes[2].set_title('Monetary (mean order £)')
plt.suptitle('RFM Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
rfm_clip = rfm.copy()
rfm_clip['monetary_clip'] = rfm['monetary'].clip(upper=rfm['monetary'].quantile(0.98))
rfm_clip['frequency_clip'] = rfm['frequency'].clip(upper=rfm['frequency'].quantile(0.98))
fig = px.scatter(
    rfm_clip, x='recency', y='frequency_clip',
    color='monetary_clip', size='monetary_clip',
    color_continuous_scale='Viridis',
    title='Recency vs Frequency (coloured by Monetary)'
)
fig.show()

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score']
def label_segment(score):
    if score >= 7: return 'Champions'
    elif score >= 5: return 'Loyal'
    elif score >= 3: return 'At Risk'
    else: return 'Lost'
rfm['segment'] = rfm['rfm_score'].apply(label_segment)
seg_counts = rfm['segment'].value_counts().reset_index()
seg_counts.columns = ['segment', 'count']
fig = px.pie(seg_counts, names='segment', values='count', title='Customer Segments by RFM Score')
fig.show()

## 5. Cohort Retention

In [ ]:
retention_matrix, cohort_size = build_cohort_matrix(df_customers)
print(f'cohort matrix shape: {retention_matrix.shape}')

In [ ]:
plt.figure(figsize=(18, 9))
sns.heatmap(
    retention_matrix.iloc[:, :13],
    annot=True, fmt='.0%', cmap='YlGnBu',
    linewidths=0.5, cbar_kws={'label': 'Retention Rate'}
)
plt.title('Monthly Cohort Retention (first 12 months)', fontsize=14)
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort Month')
plt.tight_layout()
plt.show()

In [ ]:
cohort_size_df = cohort_size.reset_index()
cohort_size_df.columns = ['cohort_month', 'cohort_size']
cohort_size_df['cohort_month'] = cohort_size_df['cohort_month'].astype(str)
fig = px.bar(cohort_size_df, x='cohort_month', y='cohort_size', title='New Customers per Cohort Month')
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 6. Category & Geographic Breakdown

In [ ]:
df_cat = df_customers.copy()
df_cat['category'] = df_cat['stock_code'].astype(str).str[:2]
top_cats = df_cat.groupby('category')['revenue'].sum().sort_values(ascending=False).head(20).reset_index()
fig = px.bar(top_cats, x='category', y='revenue', title='Top 20 Product Categories by Revenue')
fig.show()

In [ ]:
country_rev = df_customers.groupby('country')['revenue'].sum().sort_values(ascending=False).head(15).reset_index()
fig = px.bar(country_rev, x='country', y='revenue', title='Top 15 Countries by Revenue')
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 7. Engineered Feature Distributions

In [ ]:
master = build_master_customer_table(df_customers, return_features, cancel_features)
master.head()

In [ ]:
feat_cols = ['velocity_decay_ratio', 'category_hhi', 'spend_cv',
             'country_count', 'avg_items_per_order', 'return_rate', 'cancellation_rate']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(feat_cols):
    clip_val = master[col].quantile(0.99)
    axes[i].hist(master[col].clip(upper=clip_val), bins=50, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('value')

axes[-1].set_visible(False)
plt.suptitle('Engineered Feature Distributions (clipped at 99th pct)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ['recency', 'frequency', 'monetary', 'total_revenue',
                'velocity_decay_ratio', 'category_hhi', 'spend_cv',
                'country_count', 'avg_items_per_order', 'return_rate', 'cancellation_rate']

corr = master[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Phase 1 Summary ===')
print(f'unique customers : {master["customer_id"].nunique():,}')
print(f'total features   : {master.shape[1]}')
print('\nmedian RFM values:')
print(master[['recency', 'frequency', 'monetary']].median().round(2))
print('\nfeature table columns:')
print(list(master.columns))